# Dual Contour Pipeline: Rule-based + Isolation Forest

Этот ноутбук реализует двухконтурный подход без генерации большого количества CSV:

- **Контур A (rule-based на сыром датасете)**: поиск валидаторных и процедурных нарушений.
- **Контур B (Isolation Forest на очищенном датасете)**: поиск нетипичных поведенческих/процедурных паттернов.

По умолчанию ноутбук **не сохраняет** промежуточные CSV. Можно опционально сохранить **один объединённый** итоговый файл.

## 1) Импорты и конфигурация

- `EXPORT_FINAL=False` — ничего не пишем на диск.
- Если нужно выгрузить результат, включаем `EXPORT_FINAL=True`: будет сохранён только один CSV.

In [8]:
import importlib.util
import subprocess
import sys

# Ensure sklearn is available in the active notebook kernel.
if importlib.util.find_spec('sklearn') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn'])

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

DATA_PATH = '../hakaton.csv'
EXPORT_FINAL = False
FINAL_EXPORT_PATH = '../dual_contour_final_cases.csv'
RANDOM_STATE = 42


## 2) Загрузка сырых данных

In [9]:
raw = pd.read_csv(DATA_PATH, sep=';', dtype=str, low_memory=False)

for c in raw.columns:
    raw[c] = raw[c].astype(str).str.strip()
    raw[c] = raw[c].replace({'': np.nan, 'nan': np.nan})

raw['bdate_dt'] = pd.to_datetime(raw['bdate'], errors='coerce')
raw['test_date_dt'] = pd.to_datetime(raw['test_date'], errors='coerce')
raw['result_norm'] = raw['result'].str.lower().str.strip()
raw['class_num'] = pd.to_numeric(raw['class'], errors='coerce')

raw.shape

(25628, 25)

## 3) Контур A: Rule-based на сыром датасете

Фиксируем понятные правила и формируем единый `rule_score`.

In [10]:
# ключ ребенка для rule-аналитики
raw['child_key_rule'] = (
    raw[['last_name','first_name','middle_name','bdate']]
    .fillna('')
    .agg('|'.join, axis=1)
)

# базовые правила
raw['rule_missing_core'] = raw[['last_name','first_name','bdate','test_date','result']].isna().any(axis=1).astype(int)
raw['rule_invalid_result'] = (~raw['result_norm'].isin({'достаточный','недостаточный'})).astype(int)
raw['rule_test_before_birth'] = ((raw['test_date_dt'].notna()) & (raw['bdate_dt'].notna()) & (raw['test_date_dt'] < raw['bdate_dt'])).astype(int)
raw['rule_invalid_class'] = (raw['class_num'].isna() | ~raw['class_num'].between(1,11)).astype(int)

age_years = (raw['test_date_dt'] - raw['bdate_dt']).dt.days / 365.25
raw['rule_age_class_mismatch'] = (
    raw['class_num'].notna() & age_years.notna() & ((age_years < raw['class_num'] + 4) | (age_years > raw['class_num'] + 10))
).astype(int)

raw['rule_variant_nonnumeric'] = (~raw['variant'].fillna('').str.fullmatch(r'\d+')).astype(int)

# частота тестирования
raw = raw.sort_values(['child_key_rule','test_date_dt','our_number']).copy()
raw['prev_test_date_rule'] = raw.groupby('child_key_rule')['test_date_dt'].shift(1)
raw['days_since_prev_rule'] = (raw['test_date_dt'] - raw['prev_test_date_rule']).dt.days
raw['rule_freq_lt_90'] = (raw['days_since_prev_rule'].notna() & (raw['days_since_prev_rule'] < 90)).astype(int)

rule_cols = [c for c in raw.columns if c.startswith('rule_')]
raw['rule_score'] = raw[rule_cols].sum(axis=1)

rule_summary = pd.DataFrame({
    'rule': rule_cols,
    'count': [int(raw[c].sum()) for c in rule_cols]
}).assign(share_pct=lambda d: (d['count']/len(raw)*100).round(2)).sort_values('count', ascending=False)

rule_summary

,rule,count,share_pct
6,rule_freq_lt_90,1373,5.36
5,rule_variant_nonnumeric,830,3.24
4,rule_age_class_mismatch,240,0.94
2,rule_test_before_birth,1,0.00
3,rule_invalid_class,1,0.00
0,rule_missing_core,0,0.00
1,rule_invalid_result,0,0.00


## 4) Подготовка очищенной витрины для контура B

Делаем упрощённую очистку для IF:
- нормализуем ключ ребенка через `id_doc` (если есть),
- убираем точные дубли событий,
- считаем `quality_score` из rule-флагов (как вспомогательный признак).

In [11]:
clean = raw.copy()

clean['id_doc_norm'] = clean['id_doc'].fillna('').str.upper().str.replace(r'[^0-9A-ZА-Я]', '', regex=True)
fallback = clean[['last_name','first_name','middle_name','bdate']].fillna('').agg('|'.join, axis=1).str.upper()
clean['child_key_clean'] = np.where(clean['id_doc_norm'].str.len()>0, 'DOC|' + clean['id_doc_norm'], 'NAME|' + fallback)

dupe_key = ['child_key_clean','test_date','variant','class','result_norm','ogrn_naprav','ogrn_area']
clean = clean.drop_duplicates(subset=dupe_key, keep='first').copy()

clean = clean.sort_values(['child_key_clean','test_date_dt','our_number']).copy()
clean['prev_test_date_clean'] = clean.groupby('child_key_clean')['test_date_dt'].shift(1)
clean['days_since_prev_clean'] = (clean['test_date_dt'] - clean['prev_test_date_clean']).dt.days

clean.shape

(25072, 40)

## 5) Контур B: Isolation Forest на очищенном датасете

Обучаем IF без таргета, как детектор нетипичного профиля записи.

In [12]:
clean['test_month_num'] = clean['test_date_dt'].dt.month
clean['test_weekday'] = clean['test_date_dt'].dt.weekday
clean['child_attempt_no'] = clean.groupby('child_key_clean').cumcount() + 1
clean['child_total_attempts'] = clean.groupby('child_key_clean')['our_number'].transform('count')
clean['prev_rule_score'] = clean.groupby('child_key_clean')['rule_score'].shift(1)
clean['prev_result_norm'] = clean.groupby('child_key_clean')['result_norm'].shift(1)

num_features = [
    'rule_score','class_num','child_attempt_no','child_total_attempts',
    'test_month_num','test_weekday','prev_rule_score'
]
cat_features = ['result_norm','prev_result_norm','variant','ogrn_naprav','ogrn_area']

X = clean[num_features + cat_features].copy()

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
])

Xp = prep.fit_transform(X)

iforest = IsolationForest(
    contamination=0.03,
    n_estimators=500,
    max_samples=1024,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
iforest.fit(Xp)

clean['iforest_decision'] = iforest.decision_function(Xp)
clean['iforest_is_anomaly'] = np.where(iforest.predict(Xp)==-1, 1, 0)
clean['iforest_risk_score'] = -clean['iforest_decision']

q1 = clean['iforest_risk_score'].quantile(0.70)
q2 = clean['iforest_risk_score'].quantile(0.90)
clean['iforest_bucket'] = np.where(clean['iforest_risk_score'] >= q2, 'high', np.where(clean['iforest_risk_score'] >= q1, 'medium', 'low'))

clean['iforest_is_anomaly'].value_counts()

iforest_is_anomaly
0    24319
1      753
Name: count, dtype: int64

## 6) Объединение контуров A и B

Собираем единый приоритет:
- `must_check`: есть жёсткие rule-нарушения или высокий IF-риск с rule-сигналом
- `review`: IF аномалия без жёстких rule-нарушений
- `low`: остальное

In [13]:
final = clean.copy()

# Жесткие нарушения регламента
final['hard_violation'] = (
    (final['rule_freq_lt_90'] == 1) |
    (final['rule_test_before_birth'] == 1) |
    (final['rule_missing_core'] == 1)
).astype(int)

final['queue'] = np.where(
    (final['hard_violation'] == 1) | ((final['iforest_is_anomaly'] == 1) & (final['rule_score'] >= 2)),
    'must_check',
    np.where(final['iforest_is_anomaly'] == 1, 'review', 'low')
)

queue_summary = final['queue'].value_counts().rename_axis('queue').reset_index(name='count')
queue_summary['share_pct'] = (queue_summary['count']/len(final)*100).round(2)
queue_summary

,queue,count,share_pct
0,low,23545,93.91
1,must_check,853,3.40
2,review,674,2.69


## 7) Понятные причины для человека

Формируем `human_reason` прямо в итоговой таблице.

In [14]:
def explain_row(r):
    reasons=[]
    if r['rule_freq_lt_90']==1:
        reasons.append('повторное тестирование раньше 90 дней')
    if r['rule_test_before_birth']==1:
        reasons.append('дата теста раньше даты рождения')
    if r['rule_missing_core']==1:
        reasons.append('пропущены критичные поля записи')
    if r['rule_variant_nonnumeric']==1:
        reasons.append('нестандартный формат варианта теста')
    if r['rule_age_class_mismatch']==1:
        reasons.append('возможная несостыковка возраста и класса')
    if (r['iforest_is_anomaly']==1) and not reasons:
        reasons.append('нетипичный профиль записи по совокупности признаков (Isolation Forest)')
    if not reasons:
        reasons.append('существенных отклонений не выявлено')
    return '; '.join(reasons)

final['human_reason'] = final.apply(explain_row, axis=1)

final_preview_cols = [
    'our_number','test_date','child_key_clean','queue','rule_score','iforest_is_anomaly',
    'iforest_bucket','days_since_prev_clean','quality_score','human_reason'
]
final[final_preview_cols].head(20)

KeyError: "['quality_score'] not in index"

## 8) Экспорт (опционально, только один файл)

Если `EXPORT_FINAL=True`, сохраняется **один** файл `dual_contour_final_cases.csv`.
Никаких дополнительных CSV ноутбук не создаёт.

In [ ]:
if EXPORT_FINAL:
    export_cols = [
        'our_number','test_date','child_key_clean','queue','rule_score','iforest_is_anomaly','iforest_risk_score','iforest_bucket',
        'rule_freq_lt_90','rule_test_before_birth','rule_missing_core','rule_invalid_class','rule_age_class_mismatch','rule_variant_nonnumeric',
        'days_since_prev_clean','quality_score','human_reason','name_naprav','name_area'
    ]
    final[export_cols].to_csv(FINAL_EXPORT_PATH, index=False)
    print('Saved:', FINAL_EXPORT_PATH)
else:
    print('EXPORT_FINAL=False -> файл не сохраняем')

## 9) Контрольные итоги

Этот ноутбук покрывает оба контура и не плодит артефакты на диске.
Для рабочей эксплуатации можно:
- включить экспорт одного итогового файла,
- подавать его в дашборд/операционный процесс.